# Capstone: Audit and Debias an Embedding

Capstone notebook for the [Bias in Data](https://josephausterweil.github.io/probintro/ethics/bias-in-data/) chapter (Part IX).

**The audit.** A model's bias is a **prior absorbed from its training data** — the same inductive bias that lets it generalize is what makes it inherit our stereotypes. Here you measure that bias, try to remove it, and watch it leak back.

- **Part A** — implement the **Word Embedding Association Test (WEAT)** on a provided small toy embedding and measure the **effect size**. *(Caliskan, Bryson & Narayanan, 2017.)*
- **Part B** — attempt a simple **linear debiasing** (project out the gender direction) and measure *what is removed* (the WEAT drop) versus *what leaks back* through correlated words (the clustering test). *(Bolukbasi et al., 2016; Gonen & Goldberg, 2019.)*
- **Part C** *(stretch)* — show a **downstream task inheriting the bias**: a classifier trained on the embedding recovers a protected attribute it was never given. *(Buolamwini & Gebru, 2018, at toy scale.)*

The toy embedding, the cosine helper, and the target/attribute sets are provided and run end-to-end. Each `TODO` states its formula. Work top to bottom; fill the `TODO`s, then un-comment the trial cells beneath them.

> This is a **toy** that illustrates the *method* with nothing hidden — not a reproduction of the study's real numbers. Real WEAT runs on downloaded word2vec/GloVe vectors; the mechanics are identical.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

print('JAX ready:', jax.__version__)

## 1. The toy embedding and cosine similarity (complete — run me)

Twelve words as 3-D vectors `[gender, career, care]`. Coordinate 0 is a gender axis (positive for words our imaginary corpus paired with men, negative with women); coordinates 1-2 are two topic axes. In this toy corpus men's words co-occur with career talk and women's with care talk -- so **gender is entangled with topic**, exactly the trap real text embeddings fall into (and the reason Part B's debiasing will struggle).

In [ ]:
# [gender, career-topic, care-topic]
emb = {
    # attribute words: two gender poles (mostly gender, a whisper of topic)
    "he":           jnp.array([ 4.0, 0.3, 0.0]),
    "man":          jnp.array([ 3.0, 0.3, 0.0]),
    "him":          jnp.array([ 3.5, 0.3, 0.0]),
    "she":          jnp.array([-4.0, 0.0, 0.3]),
    "woman":        jnp.array([-3.0, 0.0, 0.3]),
    "her":          jnp.array([-3.5, 0.0, 0.3]),
    # target words: careers (lean male in the corpus) vs care-work (lean female)
    "engineer":     jnp.array([ 3.0, 3.0, 0.0]),
    "scientist":    jnp.array([ 2.0, 3.0, 0.0]),
    "programmer":   jnp.array([ 2.5, 3.0, 0.0]),
    "nurse":        jnp.array([-3.0, 0.0, 3.0]),
    "teacher":      jnp.array([-2.0, 0.0, 3.0]),
    "receptionist": jnp.array([-2.5, 0.0, 3.0]),
}

def cos(a, b):
    """Cosine similarity: dot product over the product of lengths (vectors chapter)."""
    return jnp.dot(a, b) / (jnp.linalg.norm(a) * jnp.linalg.norm(b))

# WEAT needs two TARGET sets to compare and two ATTRIBUTE sets to compare them against.
career = ["engineer", "scientist", "programmer"]     # target set X
care   = ["nurse", "teacher", "receptionist"]        # target set Y
male   = ["he", "man", "him"]                        # attribute set A
female = ["she", "woman", "her"]                     # attribute set B

print("sanity check -- cos(engineer, he)  =", float(cos(emb["engineer"], emb["he"])))
print("sanity check -- cos(engineer, she) =", float(cos(emb["engineer"], emb["she"])))

## 2. Part A -- implement WEAT -- TODO

For a single word `w`, its **association** is how much closer it sits (in cosine) to the attribute set `A` than to `B`:

$$s(w) = \operatorname{mean}_{a \in A}\cos(w, a) - \operatorname{mean}_{b \in B}\cos(w, b)$$

The **effect size** compares the two target sets `X`, `Y` in units of their spread (Cohen's $d$):

$$d = \frac{\operatorname{mean}_{x \in X} s(x) - \operatorname{mean}_{y \in Y} s(y)}{\operatorname{std}_{w \in X \cup Y} s(w)}$$

An effect size near $0$ means no differential association; far from $0$ means strong bias. Implement both, then un-comment the trial.

In [ ]:
def assoc(w, E, A, B):
    """s(w): mean cosine to attribute set A minus mean cosine to attribute set B."""
    # TODO:
    #   to_A = jnp.mean(jnp.array([cos(E[w], E[a]) for a in A]))
    #   to_B = jnp.mean(jnp.array([cos(E[w], E[b]) for b in B]))
    #   return to_A - to_B
    raise NotImplementedError

def weat_effect_size(E, X, Y, A, B):
    """Cohen's d between target sets X and Y for attributes A vs B."""
    # TODO:
    #   sx = jnp.array([assoc(w, E, A, B) for w in X])
    #   sy = jnp.array([assoc(w, E, A, B) for w in Y])
    #   spread = jnp.std(jnp.concatenate([sx, sy]))
    #   return (jnp.mean(sx) - jnp.mean(sy)) / (spread + 1e-12)
    raise NotImplementedError

# When both are implemented, run the audit:
# print("association s(w)  (+ leans male, - leans female):")
# for w in career + care:
#     print(f"  s({w:12s}) = {float(assoc(w, emb, male, female)):+.3f}")
# d = weat_effect_size(emb, career, care, male, female)
# print(f"WEAT effect size d = {float(d):+.3f}")   # expect a large positive number

## 3. Part B -- debias, then measure removed vs. leaked -- TODO

The obvious fix (Bolukbasi et al., 2016, in miniature): estimate the single **gender direction** and project it out of every word.

$$\hat{g} = \frac{\operatorname{mean}(A) - \operatorname{mean}(B)}{\lVert \operatorname{mean}(A) - \operatorname{mean}(B)\rVert}, \qquad v' = v - (v \cdot \hat{g})\,\hat{g}$$

Then ask **two** questions of the debiased vectors:
1. *What was removed?* -- recompute the WEAT effect size (it should drop, but by how much?).
2. *What leaked back?* -- do the originally-male words still cluster together, away from the female words? (Gonen & Goldberg, 2019.) Compare mean cosine **within** the originally-male group to mean cosine **across** groups -- a downstream model's foothold.

In [ ]:
def gender_direction(E, A, B):
    """Unit vector from the mean of attribute set A to the mean of attribute set B."""
    # TODO:
    #   g = (jnp.mean(jnp.stack([E[w] for w in A]), axis=0)
    #        - jnp.mean(jnp.stack([E[w] for w in B]), axis=0))
    #   return g / jnp.linalg.norm(g)
    raise NotImplementedError

def debias(E, g_hat):
    """Return a new embedding dict with each word's g_hat component projected out."""
    # TODO:  return {w: v - jnp.dot(v, g_hat) * g_hat for w, v in E.items()}
    raise NotImplementedError

def mean_cos(A_words, B_words, E):
    """Average cosine between every word in A_words and every word in B_words."""
    return float(jnp.mean(jnp.array([cos(E[a], E[b]) for a in A_words for b in B_words])))

# When implemented, run the before/after audit:
# g_hat = gender_direction(emb, male, female)
# emb_deb = debias(emb, g_hat)
# print(f"WEAT d  before debiasing = {float(weat_effect_size(emb,     career, care, male, female)):+.3f}")
# print(f"WEAT d  after  debiasing = {float(weat_effect_size(emb_deb, career, care, male, female)):+.3f}")
#
# male_words   = male   + career     # originally-male-leaning words
# female_words = female + care       # originally-female-leaning words
# print(f"debiased: within male-group cos = {mean_cos(male_words, male_words, emb_deb):+.3f}")
# print(f"debiased: across-group cos      = {mean_cos(male_words, female_words, emb_deb):+.3f}")
#
# Interpretation (answer in the markdown cell below):
#   - How much did WEAT actually drop? Did projecting out ONE direction fix it?
#   - Does within-group still exceed across-group? Then the bias LEAKED -- a classifier
#     could recover gender from the "debiased" vectors. Why? (Hint: gender was entangled
#     with topic; you only removed the axis you could name.)

**Your interpretation (Part B):**

*Double-click and answer here.* How much did the WEAT effect size drop after debiasing? Did removing one direction remove the bias? What does the within-vs-across comparison tell you about whether a downstream model could still recover gender -- and why does entanglement with topic make simple debiasing incomplete?

## 4. Part C (stretch) -- a downstream task inherits the bias -- TODO

The bias is not just an abstract cosine -- it flows into whatever you build *on top* of the embedding. Here the protected attribute is **gender**, which no fair system should use. Train a tiny linear classifier to recover gender from the (optionally debiased) vectors: if it succeeds, the protected attribute leaked into the representation -- the Gender Shades failure in miniature, where a model inherits a disparity for a group it was never explicitly told about.

In [ ]:
# Feature matrix and the protected label over the 12 words.
words    = list(emb.keys())
X_feat   = jnp.stack([emb[w] for w in words])                       # (12, 3)
# protected attribute we must NOT want encoded: 1 = male-leaning, 0 = female-leaning
y_gender = jnp.array([1.0 if w in (male + career) else 0.0 for w in words])

def train_logistic(feats, targets, steps=400, lr=0.1):
    """Fit P(target=1) = sigmoid(feats @ v + b) by gradient descent. Returns params."""
    # TODO:
    #   def nll(params):
    #       p = jax.nn.sigmoid(feats @ params['v'] + params['b'])
    #       return -jnp.mean(targets*jnp.log(p+1e-9) + (1-targets)*jnp.log(1-p+1e-9))
    #   params = {'v': jnp.zeros(feats.shape[1]), 'b': jnp.array(0.0)}
    #   for _ in range(steps):
    #       grads = jax.grad(nll)(params)
    #       params = jax.tree.map(lambda a, d: a - lr*d, params, grads)
    #   return params
    raise NotImplementedError

def accuracy(params, feats, targets):
    pred = jax.nn.sigmoid(feats @ params['v'] + params['b']) > 0.5
    return float(jnp.mean(pred == (targets > 0.5)))

# When implemented:
#   1. Train a GENDER classifier on the RAW embedding -- how accurately can it recover
#      the protected attribute it was never supposed to encode?
#   2. Debias (Part B), retrain on emb_deb's features -- did accuracy fall to chance
#      (0.5), or does gender still leak through the entangled topic dimensions?
#
# gp = train_logistic(X_feat, y_gender)
# print(f"gender recoverable from raw embedding: accuracy = {accuracy(gp, X_feat, y_gender):.2f}")
print("Part C: fill train_logistic, then un-comment the trial above.")

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*